# DS4420 Project - Initial Steps

Authors: Gavin Bond, Benjamin Kataoka, Preetish Paul

## Step 1: Building the Combined Dataset

We are combining the 2024 and 2025 Stack Overflow Developer Survey datasets to give our model more data to work with. Both years ask the same core questions about AI trust, so combining them roughly doubles our dataset from about 25,000 developers to around 55,000. Before combining we need to adjust three columns where the response options changed slightly between years. This dataset will serve as the base for our project, with additional preprocessing, feature engineering, and target variable manipulation within each model.

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load both survey years
file_2024 = "data/stack-overflow-developer-survey-2024/survey_results_public.csv"
file_2025 = "data/stack-overflow-developer-survey-2025/survey_results_public.csv"

df24 = pd.read_csv(file_2024, low_memory=False)
df25 = pd.read_csv(file_2025, low_memory=False)

print("2024 shape:", df24.shape)
print("2025 shape:", df25.shape)

2024 shape: (65437, 114)
2025 shape: (49191, 172)


### Filter to Professional Developers

Same filter as the POC. We only want respondents who identified as professional developers since that is the population our research question is about.

In [3]:
df24 = df24[df24["MainBranch"] == "I am a developer by profession"].copy()
df25 = df25[df25["MainBranch"] == "I am a developer by profession"].copy()

print("2024 professional developers:", df24.shape[0])
print("2025 professional developers:", df25.shape[0])

2024 professional developers: 50207
2025 professional developers: 37467


### Harmonize Columns

Three columns changed their response options between 2024 and 2025 and need to be aligned before combining. Everything else is already compatible across both years.

In [4]:
# AISelect: 2025 broke "Yes" into daily/weekly/monthly, map all back to "Yes"
df25["AISelect"] = df25["AISelect"].replace({
    "Yes, I use AI tools daily": "Yes",
    "Yes, I use AI tools weekly": "Yes",
    "Yes, I use AI tools monthly or infrequently": "Yes"
})

print("AISelect 2024:", sorted(df24["AISelect"].dropna().unique()))
print("AISelect 2025:", sorted(df25["AISelect"].dropna().unique()))

AISelect 2024: ["No, and I don't plan to", 'No, but I plan to soon', 'Yes']
AISelect 2025: ["No, and I don't plan to", 'No, but I plan to soon', 'Yes']


In [5]:
# OrgSize: 2024 had "2 to 9" and "10 to 19" separately, 2025 merged them to "Less than 20 employees"
df24["OrgSize"] = df24["OrgSize"].replace({
    "2 to 9 employees": "Less than 20 employees",
    "10 to 19 employees": "Less than 20 employees"
})

print("OrgSize 2024:", sorted(df24["OrgSize"].dropna().unique()))
print("OrgSize 2025:", sorted(df25["OrgSize"].dropna().unique()))

OrgSize 2024: ['1,000 to 4,999 employees', '10,000 or more employees', '100 to 499 employees', '20 to 99 employees', '5,000 to 9,999 employees', '500 to 999 employees', 'I don’t know', 'Just me - I am a freelancer, sole proprietor, etc.', 'Less than 20 employees']
OrgSize 2025: ['1,000 to 4,999 employees', '10,000 or more employees', '100 to 499 employees', '20 to 99 employees', '5,000 to 9,999 employees', '500 to 999 employees', 'I don’t know', 'Just me - I am a freelancer, sole proprietor, etc.', 'Less than 20 employees']


In [6]:
# RemoteWork: 2025 split "Hybrid" into several sub-options, map all back to "Hybrid (some remote, some in-person)"
df25["RemoteWork"] = df25["RemoteWork"].replace({
    "Hybrid (some in-person, leans heavy to flexibility)": "Hybrid (some remote, some in-person)",
    "Hybrid (some remote, leans heavy to in-person)": "Hybrid (some remote, some in-person)",
    "Your choice (very flexible, you can come in when you want or just as needed)": "Hybrid (some remote, some in-person)"
})

print("RemoteWork 2024:", sorted(df24["RemoteWork"].dropna().unique()))
print("RemoteWork 2025:", sorted(df25["RemoteWork"].dropna().unique()))

RemoteWork 2024: ['Hybrid (some remote, some in-person)', 'In-person', 'Remote']
RemoteWork 2025: ['Hybrid (some remote, some in-person)', 'In-person', 'Remote']


### Select Shared Columns and Combine

We select only the columns we need for the model, tag each row with its survey year, and then concatenate the two datasets.

In [7]:
# Columns used in the model
feature_cols = [
    "YearsCode", "WorkExp",
    "Age", "DevType", "EdLevel", "OrgSize",
    "AISelect", "AISent", "AIThreat", "RemoteWork",
    "AIComplex", "ICorPM"
]
target_col = "AIAcc"

keep_cols = feature_cols + [target_col]

# Tag each year and combine
df24["year"] = 2024
df25["year"] = 2025

df_combined = pd.concat(
    [df24[keep_cols + ["year"]], df25[keep_cols + ["year"]]],
    ignore_index=True
)

print("Combined shape:", df_combined.shape)
print("Year breakdown:")
print(df_combined["year"].value_counts())

Combined shape: (87674, 14)
Year breakdown:
year
2024    50207
2025    37467
Name: count, dtype: int64


### Filter to Rows with AIAcc and Verify

We drop rows where the target variable is missing since we cannot train or evaluate on those. We then do a final check to make sure the combined dataset looks clean.

In [8]:
# Drop rows with missing target
df_combined = df_combined[df_combined[target_col].notna()].copy()

print("Combined shape after dropping missing AIAcc:", df_combined.shape)
print()
print("AIAcc distribution:")
print(df_combined[target_col].value_counts())
print()
print("Year breakdown after filter:")
print(df_combined["year"].value_counts())

Combined shape after dropping missing AIAcc: (55118, 14)

AIAcc distribution:
AIAcc
Somewhat trust                19144
Neither trust nor distrust    13582
Somewhat distrust             13536
Highly distrust                7490
Highly trust                   1366
Name: count, dtype: int64

Year breakdown after filter:
year
2024    29379
2025    25739
Name: count, dtype: int64


In [9]:
# Verify that the columns are properly cleaned and combined correctly
for col in ["AISelect", "OrgSize", "RemoteWork"]:
    print(f"--- {col} ---")
    print(df_combined[col].dropna().unique())
    print()

--- AISelect ---
<StringArray>
['Yes', 'No, and I don't plan to', 'No, but I plan to soon']
Length: 3, dtype: str

--- OrgSize ---
<StringArray>
[                              '100 to 499 employees',
                             'Less than 20 employees',
                                 '20 to 99 employees',
                           '5,000 to 9,999 employees',
                           '1,000 to 4,999 employees',
 'Just me - I am a freelancer, sole proprietor, etc.',
                                       'I don’t know',
                           '10,000 or more employees',
                               '500 to 999 employees']
Length: 9, dtype: str

--- RemoteWork ---
<StringArray>
['Remote', 'Hybrid (some remote, some in-person)', 'In-person']
Length: 3, dtype: str



### Export Combined Dataset

Now that the dataset is clean and verified we export it to a CSV so other notebooks can load it directly without needing to redo all the preprocessing steps.

In [10]:
# Export to CSV for use in other notebooks
df_combined.to_csv("data/combined_survey.csv", index=False)
print("Exported to data/combined_survey.csv")
print("Shape:", df_combined.shape)

Exported to data/combined_survey.csv
Shape: (55118, 14)
